In [ ]:
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
# Model names (without .csv) or full paths.
# Prefix with 'baseline:' to pull from results/whisper_baseline/whisper_baseline.csv
MODEL_A = 'baseline:whisper'
MODEL_B = 'bridge_dtw_eps'

# Columns to show in the comparison table.
# Use None to show all metric columns.
SHOW_COLS = ['utt_wer']   # or None

# Optional filters (set to None to skip)
FILTER_L1      = None   # e.g. 'Hindi' — filters both tables to this L1
FILTER_SPEAKER = None   # e.g. 'THV'

# Sort by this column in the final comparison (suffix _a or _b)
SORT_BY = 'utt_wer_a'
SORT_ASC = False  # False = worst first

# ── Path resolution ───────────────────────────────────────────────────────────
ROOT = Path("/vol/gpudata/tsv22-fyp/accent-robust-asr")
BRIDGE_DIR   = Path(f'{ROOT}/results/bridge_eval')
STEERING_DIR = Path(f'{ROOT}/results/e2_steering')
BASELINE_DIR = Path(f'{ROOT}/results/whisper_baseline')

def resolve(name: str) -> Path:
    if name.startswith('baseline:'):
        return BASELINE_DIR / 'whisper_baseline.csv'
    p = Path(name)
    if p.suffix == '.csv' and p.exists():
        return p
    for d in [BRIDGE_DIR, STEERING_DIR]:
        candidate = d / f'{name}.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find CSV for {name!r}')

path_a = resolve(MODEL_A)
path_b = resolve(MODEL_B)
print(f'A: {path_a}')
print(f'B: {path_b}')

A: /vol/gpudata/tsv22-fyp/accent-robust-asr/results/whisper_baseline/whisper_baseline.csv
B: /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_eps.csv


In [ ]:
def load(path: Path, label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # normalise column names across different eval scripts
    renames = {'wer': 'utt_wer', 'mer': 'utt_mer', 'per': 'utt_per',
               'whisper_pred': 'prediction', 'whisper_pred_norm': 'prediction_norm'}
    df = df.rename(columns={k: v for k, v in renames.items() if k in df.columns})
    df['_label'] = label
    return df

da = load(path_a, MODEL_A)
db = load(path_b, MODEL_B)

print(f'A: {len(da)} rows   cols: {list(da.columns)}')
print(f'B: {len(db)} rows   cols: {list(db.columns)}')

A: 31395 rows   cols: ['speaker', 'utterance_id', 'l1', 'speaker_type', 'bridge_split', 'text', 'prediction', 'reference_norm', 'prediction_norm', 'utt_wer', 'utt_mer', 'utt_per', '_label']
B: 350 rows   cols: ['utterance_id', 'speaker', 'l1', 'domain', 'wav_path', 'text', 'prediction', 'reference_norm', 'prediction_norm', 'ref_num_words', 'utt_wer', 'utt_mer', 'utt_per', '_label']


In [ ]:
def apply_filters(df):
    if FILTER_L1 and 'l1' in df.columns:
        df = df[df['l1'] == FILTER_L1]
    if FILTER_SPEAKER and 'speaker' in df.columns:
        df = df[df['speaker'] == FILTER_SPEAKER]
    return df

da = apply_filters(da)
db = apply_filters(db)

# Determine metric columns (everything after the join keys / text cols)
NON_METRIC = {'utterance_id', 'speaker', 'l1', 'domain', 'wav_path', 'text',
              'prediction', 'reference_norm', 'prediction_norm', 'speaker_type',
              'bridge_split', '_label'}
metric_cols_a = [c for c in da.columns if c not in NON_METRIC]
metric_cols_b = [c for c in db.columns if c not in NON_METRIC]
shared_metrics = [c for c in metric_cols_a if c in metric_cols_b]
cols_to_show = SHOW_COLS if SHOW_COLS else shared_metrics

# Carry over text for browsing
keep_a = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in da.columns else []) + \
         ['text'] + cols_to_show + \
         (['prediction_norm'] if 'prediction_norm' in da.columns else [])
keep_b = ['utterance_id', 'speaker'] + cols_to_show + \
         (['prediction_norm'] if 'prediction_norm' in db.columns else [])

merged = da[keep_a].merge(
    db[[c for c in keep_b if c in db.columns]],
    on=['utterance_id', 'speaker'],
    suffixes=('_a', '_b'),
    how='inner'
)

# Delta columns
for c in cols_to_show:
    ca, cb = f'{c}_a', f'{c}_b'
    if ca in merged.columns and cb in merged.columns:
        merged[f'{c}_delta'] = merged[cb] - merged[ca]
        
if SORT_BY and SORT_BY in merged.columns:
    merged = merged.sort_values(SORT_BY, ascending=SORT_ASC).reset_index(drop=True)

print(f'Joined: {len(merged)} utterances')
merged.head(10)

Joined: 350 utterances


,utterance_id,speaker,l1,text,utt_wer_a,utt_per_a,prediction_norm_a,utt_wer_b,utt_per_b,prediction_norm_b,utt_wer_delta,utt_per_delta
0,arctic_b0328,BWC,Chinese,Change chairs Daylight commanded,1.000000,0.190476,change chairs they like command it,1.000000,0.285714,change chairs they like to mandate,0.000000,0.095238
1,arctic_a0162,HQTV,Vietnamese,That's the sub foreman explained Thorpe,1.000000,0.481481,thus the suffering man is planned thought,0.833333,0.444444,thus the sephoman is planned thought,-0.166667,-0.037037
2,arctic_a0458,HQTV,Vietnamese,The stout wood was crushed like an eggshell,0.875000,0.370370,the starwood will crush light and ash shell,0.625000,0.259259,the starwoods were crushed like an ash shell,-0.250000,-0.111111
3,arctic_a0089,HQTV,Vietnamese,The night glow was treacherous to shoot by,0.875000,0.480000,the nycloid was stretcher as you should bite,0.750000,0.280000,the nyclo was stretcherous you should buy,-0.125000,-0.200000
4,arctic_b0268,HJK,Korean,Saxon nodded and the boy frowned,0.833333,0.434783,sex and knotted and a boyfriend frowned,0.500000,0.173913,sex and knotted and the boy frowned,-0.333333,-0.260870
5,arctic_b0350,BWC,Chinese,Stand off butcher and baker and all the rest,0.777778,0.448276,then ill put her and bake her art all the rest,0.666667,0.620690,then ill make her all the rest,-0.111111,0.172414
6,arctic_b0183,HQTV,Vietnamese,No I did not fall among thieves,0.714286,0.600000,no i did not throw in the wrong tip,0.714286,0.600000,no i did not throw in the wrong place,0.000000,0.000000
7,arctic_a0448,ZHAA,Arabic,I was Hump cabin boy on the schooner Ghost,0.666667,0.285714,i was humped kept in boy and this cooner ghost,29.444444,28.178571,i was humped i was humped i was humped i was humped i was humped i was humpe...,28.777778,27.892857
8,arctic_a0458,BWC,Chinese,The stout wood was crushed like an eggshell,0.625000,0.222222,the style was crashed like an egg shell,0.375000,0.333333,the style was crushed like an axel,-0.250000,0.111111
9,arctic_a0477,BWC,Chinese,Wada Louis and the steward are servants of Asiatic breed,0.600000,0.325000,what are lewis and stewart are sourd of asiatic breed,0.500000,0.250000,what a lewis and the steward are sourd of asianic breed,-0.100000,-0.075000


In [ ]:
import numpy as np

print(f'Model A: {MODEL_A}')
print(f'Model B: {MODEL_B}')
print(f'Utterances: {len(merged)}\n')

for c in cols_to_show:
    ca, cb, cd = f'{c}_a', f'{c}_b', f'{c}_delta'
    if ca not in merged.columns or cb not in merged.columns:
        continue
    a_mean = merged[ca].mean()
    b_mean = merged[cb].mean()
    d_mean = merged[cd].mean() if cd in merged.columns else float('nan')
    print(f'{c}:')
    print(f'  A ({MODEL_A}): {a_mean:.4f}')
    print(f'  B ({MODEL_B}): {b_mean:.4f}')
    print(f'  delta A−B:    {d_mean:+.4f}  ({"A worse" if d_mean > 0 else "A better"})')
    print()

Model A: baseline:whisper
Model B: bridge_dtw_eps
Utterances: 350

utt_wer:
  A (baseline:whisper): 0.1624
  B (bridge_dtw_eps): 0.7478
  delta A−B:    +0.5854  (A worse)

utt_per:
  A (baseline:whisper): 0.0778
  B (bridge_dtw_eps): 0.5520
  delta A−B:    +0.4742  (A worse)



In [ ]:
# Breakdown by L1 (if available)
if 'l1' in merged.columns:
    rows = []
    for l1, g in merged.groupby('l1'):
        row = {'l1': l1, 'n': len(g)}
        for c in cols_to_show:
            ca, cb = f'{c}_a', f'{c}_b'
            if ca in g.columns: row[f'{c}_A'] = g[ca].mean()
            if cb in g.columns: row[f'{c}_B'] = g[cb].mean()
            if ca in g.columns and cb in g.columns:
                row[f'{c}_Δ'] = g[ca].mean() - g[cb].mean()
        rows.append(row)
    pd.DataFrame(rows).set_index('l1').round(4)

In [ ]:
# ── Browse individual utterances ──────────────────────────────────────────────
# Change slice / filter here to zoom in
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)

view = merged.copy()

# Uncomment to filter:
# view = view[view['l1'] == 'Hindi']
# view = view[view['utt_wer_delta'] > 0.5]   # A much worse than B
# view = view[view['utt_wer_delta'] < -0.5]  # A much better than B

text_cols  = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in view.columns else []) + ['text']
pred_cols  = [c for c in view.columns if 'prediction_norm' in c]
metric_disp = [c for c in view.columns if any(c.startswith(m) for m in cols_to_show)]

view[text_cols + pred_cols + metric_disp].head(30)

,utterance_id,speaker,l1,text,prediction_norm_a,prediction_norm_b,utt_wer_a,utt_per_a,utt_wer_b,utt_per_b,utt_wer_delta,utt_per_delta
0,arctic_b0328,BWC,Chinese,Change chairs Daylight commanded,change chairs they like command it,change chairs they like to mandate,1.000000,0.190476,1.000000,0.285714,0.000000,-0.095238
1,arctic_a0162,HQTV,Vietnamese,That's the sub foreman explained Thorpe,thus the suffering man is planned thought,thus the sephoman is planned thought,1.000000,0.481481,0.833333,0.444444,0.166667,0.037037
2,arctic_a0458,HQTV,Vietnamese,The stout wood was crushed like an eggshell,the starwood will crush light and ash shell,the starwoods were crushed like an ash shell,0.875000,0.370370,0.625000,0.259259,0.250000,0.111111
3,arctic_a0089,HQTV,Vietnamese,The night glow was treacherous to shoot by,the nycloid was stretcher as you should bite,the nyclo was stretcherous you should buy,0.875000,0.480000,0.750000,0.280000,0.125000,0.200000
4,arctic_b0268,HJK,Korean,Saxon nodded and the boy frowned,sex and knotted and a boyfriend frowned,sex and knotted and the boy frowned,0.833333,0.434783,0.500000,0.173913,0.333333,0.260870
5,arctic_b0350,BWC,Chinese,Stand off butcher and baker and all the rest,then ill put her and bake her art all the rest,then ill make her all the rest,0.777778,0.448276,0.666667,0.620690,0.111111,-0.172414
6,arctic_b0183,HQTV,Vietnamese,No I did not fall among thieves,no i did not throw in the wrong tip,no i did not throw in the wrong place,0.714286,0.600000,0.714286,0.600000,0.000000,0.000000
7,arctic_a0448,ZHAA,Arabic,I was Hump cabin boy on the schooner Ghost,i was humped kept in boy and this cooner ghost,i was humped i was humped i was humped i was humped i was humped i was humpe...,0.666667,0.285714,29.444444,28.178571,-28.777778,-27.892857
8,arctic_a0458,BWC,Chinese,The stout wood was crushed like an eggshell,the style was crashed like an egg shell,the style was crushed like an axel,0.625000,0.222222,0.375000,0.333333,0.250000,-0.111111
9,arctic_a0477,BWC,Chinese,Wada Louis and the steward are servants of Asiatic breed,what are lewis and stewart are sourd of asiatic breed,what a lewis and the steward are sourd of asianic breed,0.600000,0.325000,0.500000,0.250000,0.100000,0.075000
